# Evaluation Script: Insecure loadUrl Navigations
This script produces results for Section 5.4 Insecure `loadUrl` Navigations

### Connect to the MongoDB

In [ ]:
import tqdm
import pymongo

import pandas as pd

from urllib.parse import urlparse

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["webview"]
network_logs_collection = db["network_logs"]
dynamic_api_calls_collection = db["dynamic_api_calls"]

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Retrieve stats on `loadUrl` calls

Results for "Usage and Loaded Schemes" (1/2)

In [2]:
# apps using loadUrl
amount_apps_using_webview = len(dynamic_api_calls_collection.distinct("source_package_name"))
print_latex_macro("appsUsingWebView", f"{amount_apps_using_webview:,}")
amount_apps_using_load_url = len(dynamic_api_calls_collection.distinct("source_package_name", {"api": "LOAD_URL"}))
print_latex_macro("appsUsingLoadUrl", f"{amount_apps_using_load_url:,}")
print_latex_macro("appsUsingLoadUrlPercentage", f"{(amount_apps_using_load_url/amount_apps_using_webview)*100:.2f}")

\newcommand{\appsUsingWebView}{25,186}
\newcommand{\appsUsingLoadUrl}{21,831}
\newcommand{\appsUsingLoadUrlPercentage}{86.68}


### Retrieve commonly-used schemes

Results for "Usage and Loaded Schemes" (2/2) and Local Address HTTP URLs (1/2)

In [4]:
results = dynamic_api_calls_collection.find({
    "api": "LOAD_URL"
})

package_names = set()
total_package_names = set()
http_localhost_package_names = set()

all_http_package_names = set()

schemes = {}
for result in tqdm.tqdm(results, desc="Analyzing LOAD_URL schemes", total=dynamic_api_calls_collection.count_documents({"api": "LOAD_URL"})):
    
    package_name = result.get("source_package_name")
    params = result.get("params", {})
    url = params[0]
    try:
        parsed_url = urlparse(url)
    except Exception as e:
        continue
    scheme = parsed_url.scheme
    if scheme is None:
        scheme = "\\code{null}"
    if scheme == "" or scheme == b"":
        scheme = "empty string"
    total_package_names.add(package_name)
    # check if localhost

    if scheme == "http":
        all_http_package_names.add(package_name)

    if (parsed_url.hostname == "localhost" or parsed_url.hostname == "127.0.0.1" or parsed_url.hostname == "0.0.0.0"):
        if scheme == "http":
            http_localhost_package_names.add(package_name)
        scheme = f"\\code{{{scheme}}} (local)"
    elif scheme == "http":
        package_names.add(package_name)

    if (url == "about:blank" or url == "about:blank/"):
        scheme = "about:blank"

    
    if not scheme.startswith("\\"):
        scheme = scheme.lower()
        scheme = f"\\code{{{scheme}}}"
    
    if scheme not in schemes:
        schemes[scheme] = set()
    schemes[scheme].add(package_name)

print_latex_macro("appsLoadingTopLevelAllHTTP", f"{len(all_http_package_names):,}")
print_latex_macro("appsLoadingTopLevelAllHTTPPercentage", f"{(len(all_http_package_names)/amount_apps_using_webview)*100:.2f}")
print_latex_macro("appsLoadingTopLevelNonLocalhostHTTPContent", f"{len(package_names):,}")
print_latex_macro("appsLoadingTopLevelNonLocalhostHTTPContentPercentage", f"{(len(package_names)/amount_apps_using_webview)*100:.2f}")
print_latex_macro("appsLoadingTopLevelLocalhostHTTPContent", f"{len(http_localhost_package_names):,}")
print_latex_macro("appsLoadingTopLevelLocalhostHTTPContentPercentage", f"{(len(http_localhost_package_names)/amount_apps_using_webview)*100:.2f}")

Analyzing LOAD_URL schemes: 100%|██████████| 689924/689924 [02:03<00:00, 5592.34it/s] 

\newcommand{\appsLoadingTopLevelAllHTTP}{2,260}
\newcommand{\appsLoadingTopLevelAllHTTPPercentage}{8.97}
\newcommand{\appsLoadingTopLevelNonLocalhostHTTPContent}{1,354}
\newcommand{\appsLoadingTopLevelNonLocalhostHTTPContentPercentage}{5.38}
\newcommand{\appsLoadingTopLevelLocalhostHTTPContent}{925}
\newcommand{\appsLoadingTopLevelLocalhostHTTPContentPercentage}{3.67}


### Create a table with commonly-used URL schemes

Creates Table 3

In [5]:
# Create a table with the loaded top-level http content package namese
latex_table = ""
table_header = r"""
\begin{table}[ht]
\centering
\footnotesize
\caption{Top 8 URL schemes loaded via \code{loadUrl}. Local refers to \code{localhost}, \code{127.0.0.1}, and \code{0.0.0.0}.}
\label{tab:url-schemes}
\resizebox{\columnwidth}{!}{% % Resizes everything to fit the column width
\begin{tabular}{ll| ll}
\toprule
\textbf{Scheme} & \textbf{\# Apps} & \textbf{Scheme} & \textbf{\# Apps} \\
\midrule
"""
latex_table += table_header

# Sort schemes by count (descending) and take top 10
top10 = sorted(schemes.items(), key=lambda x: len(x[1]), reverse=True)[:8]

# Split into two columns (5 each)
left = top10[:4]
right = top10[4:]

# Fill the table rows
for i in range(4):
    left_scheme, left_count = left[i]
    right_scheme, right_count = right[i]
    latex_table += f"{left_scheme} & {len(left_count):,} ({(len(left_count)/amount_apps_using_webview)*100:.2f}\\%) & {right_scheme} & {len(right_count):,} ({(len(right_count)/amount_apps_using_webview)*100:.2f}\\%) \\\\\n"

latex_table += r"""
\bottomrule
\end{tabular}
}
\end{table}
"""

print(latex_table)


\begin{table}[ht]
\centering
\footnotesize
\caption{Top 8 URL schemes loaded via \code{loadUrl}. Local refers to \code{localhost}, \code{127.0.0.1}, and \code{0.0.0.0}.}
\label{tab:url-schemes}
\resizebox{\columnwidth}{!}{% % Resizes everything to fit the column width
\begin{tabular}{ll| ll}
\toprule
\textbf{Scheme} & \textbf{\# Apps} & \textbf{Scheme} & \textbf{\# Apps} \\
\midrule
\code{https} & 19,563 (77.67\%) & \code{http} & 1,354 (5.38\%) \\
\code{about:blank} & 10,796 (42.87\%) & \code{http} (local) & 925 (3.67\%) \\
\code{file} & 4,399 (17.47\%) & \code{empty string} & 835 (3.32\%) \\
\code{javascript} & 4,075 (16.18\%) & \code{https} (local) & 445 (1.77\%) \\

\bottomrule
\end{tabular}
}
\end{table}



### Retrieve `loadUrl` calls without a scheme

Results for "Missing Protocol"

**Attention: The amount of apps where DNS lookups succeed may differ!**

In [6]:
### check if there are apps that load a website without a scheme
import requests
import socket


results = dynamic_api_calls_collection.find({
    "api": "LOAD_URL"
})

package_names = set()
total_package_names = set()

def probe_url(url):
    # 1. Try to request the URL with http scheme
    
    try:
        # We use a timeout to prevent the script from hanging
        response = requests.get(test_url, timeout=5)
        return True
    except requests.exceptions.RequestException as e:
        pass

    # 2. Perform a DNS Lookup for the URL
    try:
        # gethostbyname returns the IPv4 address
        ip_address = socket.gethostbyname(url[7:])  # strip "http://"
        return True
    except socket.gaierror:
        pass
    
    return False

schemes = {}

package_names_none_url = set()

packages_http_no_scheme_loaded = set() # collects the package_names for those where the DNS could be resolved
packages_http_no_scheme_not_loaded = set() # collects the package_names for those where the DNS could not be resolved
packages_http_no_scheme_not_loaded_urls = set() # collects the (package_name, url) tuple for those where the DNS could not be resolved

for result in tqdm.tqdm(results, desc="Analyzing LOAD_URL schemes", total=dynamic_api_calls_collection.count_documents({"api": "LOAD_URL"})):
    package_name = result.get("source_package_name")
    params = result.get("params", {})
    url = params[0]
    
    # parse the url and check if it is a valid web url
    if not url:
        package_names_none_url.add(package_name)
        continue
    
    try: 
        parsed_url = urlparse(url)
    except Exception as e:
        packages_http_no_scheme_not_loaded.add(package_name)
        packages_http_no_scheme_not_loaded_urls.add((package_name, url))
        continue
    scheme = parsed_url.scheme

    if "://" in url:
        continue
    
    if scheme != "":
        continue
    
    path = parsed_url.path
    if path == "":
        continue
    
    netloc = parsed_url.netloc
    
    path = parsed_url.path
    if path.startswith("/"):
        continue
    
    # try to request the url with http scheme
    test_url = f"http://{url}"
    
    try:
        r = probe_url(test_url)
    except Exception as e:
        packages_http_no_scheme_not_loaded.add(package_name)
        packages_http_no_scheme_not_loaded_urls.add((package_name, url))
        continue
    if r:
        packages_http_no_scheme_loaded.add(package_name)
    else:
        packages_http_no_scheme_not_loaded.add(package_name)
        packages_http_no_scheme_not_loaded_urls.add((package_name, url))
        

print_latex_macro("appsLoadingNoneUrl", f"{len(package_names_none_url):,}")
print_latex_macro("appsLoadingHTTPContentWithoutSchemeLoaded", f"{len(packages_http_no_scheme_loaded):,}")
print_latex_macro("appsLoadingHTTPContentWithoutSchemeNotLoaded", f"{len(packages_http_no_scheme_not_loaded):,}")
print_latex_macro("appsLoadingHTTPContentWithoutSchemeTotal", f"{len(packages_http_no_scheme_loaded) + len(packages_http_no_scheme_not_loaded):,}")


Analyzing LOAD_URL schemes: 100%|██████████| 689924/689924 [03:36<00:00, 3184.77it/s] 

\newcommand{\appsLoadingNoneUrl}{669}
\newcommand{\appsLoadingHTTPContentWithoutSchemeLoaded}{21}
\newcommand{\appsLoadingHTTPContentWithoutSchemeNotLoaded}{46}
\newcommand{\appsLoadingHTTPContentWithoutSchemeTotal}{67}


### Retrieve apps that load `0.0.0.0`

Produces results for "Local Address HTTP URLs" (2/2)

In [5]:
results = dynamic_api_calls_collection.find({
    "api": "LOAD_URL",
    "params.0": {"$regex": r"^http://(0\.0\.0\.0)([:/]|$)"}
})
amount_apps_loading_0000 = len(results.distinct("source_package_name"))
print_latex_macro("appsLoadingZeroZeroZeroZero", f"{amount_apps_loading_0000:,}")

\newcommand{\appsLoadingZeroZeroZeroZero}{608}
